# AI Coder on Google Colab
<!-- Author: Xin Zhou (Student ID: 42742822) -->

Runs the multi-agent MCP code generator (SWE 270P final project) in Colab.

**Before you start**
1. Choose a model backend in step 3: **Gemini** (fast, needs an API key) or **Ollama** (open-source, needs a GPU runtime: *Runtime → Change runtime type → T4 GPU*).
2. For Gemini: click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GEMINI_API_KEY` with your key from https://aistudio.google.com/apikey, and turn on *Notebook access*.

Run the cells from top to bottom.

## 1. Get the project
Upload `ai_coder.zip` when the file picker appears (skipped if the folder already exists).

In [ ]:
import os, zipfile
from google.colab import files

os.chdir('/content')
if not os.path.isdir('/content/ai_coder/ai_coder'):
    uploaded = files.upload()                      # choose ai_coder.zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as zf:
        zf.extractall('/content')
%cd /content/ai_coder
!ls

## 2. Install the Python packages

In [ ]:
!pip install -q -r requirements.txt
!python --version

## 3. Choose the model backend
Set `BACKEND` to `"gemini"`, `"ollama"` or `"mock"` (offline canned answers, only for checking that everything runs).

For Gemini, run this cell once with `GEMINI_MODEL = ""` to print the available models, then put a *flash* model id in `GEMINI_MODEL` and run it again.

In [ ]:
BACKEND = "gemini"        # "gemini", "ollama" or "mock"
GEMINI_MODEL = ""         # a model id containing "flash" from the list printed below
OLLAMA_MODEL = "qwen2.5-coder:7b"
AUTHOR_NAME = "Xin Zhou & Yuxiao Xiao"
STUDENT_ID = "42742822 & 62185048"

import json, urllib.request

settings = {"AUTHOR_NAME": AUTHOR_NAME, "STUDENT_ID": STUDENT_ID, "MAX_FIX_ROUNDS": "2"}
if BACKEND == "gemini":
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
    if not GEMINI_MODEL:
        req = urllib.request.Request("https://generativelanguage.googleapis.com/v1beta/openai/models",
                                     headers={"Authorization": f"Bearer {key}"})
        models = [m["id"] for m in json.load(urllib.request.urlopen(req))["data"]]
        print("Models with 'flash' in the name:")
        print("\n".join(m for m in models if "flash" in m))
        raise SystemExit("Copy one id into GEMINI_MODEL and run this cell again.")
    settings.update({"LLM_PROVIDER": "openai",
                     "LLM_BASE_URL": "https://generativelanguage.googleapis.com/v1beta/openai/",
                     "LLM_API_KEY": key, "LLM_MODEL": GEMINI_MODEL.removeprefix("models/"),
                     "LLM_REASONING_EFFORT": "low"})
elif BACKEND == "ollama":
    settings.update({"LLM_PROVIDER": "ollama", "LLM_MODEL": OLLAMA_MODEL, "LLM_NUM_CTX": "16384"})
else:
    settings.update({"LLM_PROVIDER": "mock"})

with open(".env", "w") as fh:                     # read by ai_coder/config.py
    fh.writelines(f"{k}={v}\n" for k, v in settings.items())
print("Wrote .env for backend:", BACKEND)

### 3b. Only for Ollama: install it and download the model (needs a GPU runtime, about 5 minutes)

In [ ]:
if BACKEND == "ollama":
    import subprocess, time
    !apt-get -qq install -y zstd pciutils lshw > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                                   stderr=subprocess.STDOUT)
    time.sleep(5)
    !ollama pull {OLLAMA_MODEL}
    !nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
else:
    print("Skipped (backend is", BACKEND + ")")

## 4. Check the installation (offline tests, about 15 s)

In [ ]:
!LLM_PROVIDER=mock python -m pytest -q

## 5. Generate the Customer Feedback Tracker from the command line
Shows every agent's progress, the test result and the LLM usage (API calls and tokens).

In [ ]:
!python run_cli.py --spec specs/customer_feedback_tracker.md

## 6. Open the web UI (for the demo)
Starts the UI in the background and opens it through Colab's proxy. If no window opens, click the link printed below.

In [ ]:
import subprocess, time
from google.colab.output import eval_js, serve_kernel_port_as_window

PORT = 8050
ui_proc = subprocess.Popen(["python", "run_ui.py", "--port", str(PORT)],
                           stdout=open("/content/ui.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
print("UI link:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
serve_kernel_port_as_window(PORT)

If the UI does not respond, look at its log:

In [ ]:
!tail -n 30 /content/ui.log

## 7. Download the results
Each run is in `output/<job_id>/`: the generated project, its ZIP, spec, design, test output and `usage_report.json`.

In [ ]:
import glob, os, shutil
runs = sorted(glob.glob("/content/ai_coder/output/2*"))
print("\n".join(os.path.basename(r) for r in runs) or "No runs yet")
if runs:
    latest = runs[-1]
    shutil.make_archive("/content/latest_run", "zip", latest)
    files.download("/content/latest_run.zip")

## Notes
- Colab deletes everything when the session ends (including the Ollama model): download your results first.
- Keep the API key in Colab Secrets. Never paste it into a cell you share or commit.
- `asyncio.run` cannot run inside a notebook cell, which is why the pipeline is started with `!python ...` (a separate process).